# Notebook 23 — contact sheets for the `strict_v1`-CORRECT spot check

The 2026-08-11 human audit read all 104 of Qwen's `genuinely_wrong` items and
found ~74% of them to be the scoring pipeline rather than the model. That
correction is **one-sided**: only items the frozen rule called *wrong* were
looked at, so only false *negatives* could be found. A reviewer's first
question is the obvious one —

> *You only looked for false negatives. Did you look for false positives?*

This notebook builds the sheets that answer it. It renders the 40 randomly
drawn items from `reference/audit/spotcheck_40_qwen_strict_v1_correct_20260811.csv`
— items `strict_v1` scored **CORRECT** — so they can be read for false passes.

**4 of the 40 are already-known `false_pass_removed` items** (31, 117, 239,
294), marked in the captions. They are the calibration check: if the human
pass catches them, the method works; if it misses them, the resulting rate
should not be trusted.

Selection was random with a fixed seed (`20260811`), never by hand, so what
comes out is an estimate rather than a collection of interesting cases.

No GPU. Reads the dataset and one results CSV; runs no generation.

In [1]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and an existing results CSV, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.plotting
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy. Fix before trusting output.")
print("SymPy LaTeX parser OK")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.6 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot
SymPy LaTeX parser OK


In [2]:
import pandas as pd

RUN_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
SPOTCHECK = "repo/reference/audit/spotcheck_40_qwen_strict_v1_correct_20260811.csv"
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5

run = pd.read_csv(f"{RESULTS_DIR}/{RUN_CSV}")
spot = pd.read_csv(SPOTCHECK)
print(f"run       {len(run)} rows, model={run['model_id'].unique().tolist()}")
print(f"spotcheck {len(spot)} items, "
      f"{int(spot['known_false_pass'].sum())} known false passes")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Checking the question text pins the
# row-to-image mapping every caption below depends on -- attaching a caption
# to the wrong page would make the whole audit worse than useless.
assert len(sample) == len(run), f"{len(sample)} items vs {len(run)} rows"
bad = [i for i in range(len(run))
       if sample[i]["orig_q"].strip() != str(run.iloc[i]["orig_q"]).strip()]
assert not bad, (
    f"{len(bad)} rows where the rebuilt sample's question does not match the "
    f"CSV's (first: {bad[:5]}). Images would be attached to the wrong rows.")
print("\nsample order matches the CSV on all 300 rows -- images are index-aligned")

# Every spot-check item must actually be one strict_v1 called correct.
scored = pilot.rescore.rescore_run(run, "strict_v1", progress=True)
correct_idx = set(scored.index[scored["transcription_correct"].astype(bool)])
assert set(spot["item"]) <= correct_idx, (
    "spot-check contains items strict_v1 did NOT score correct: "
    f"{sorted(set(spot['item']) - correct_idx)}")
print(f"all {len(spot)} spot-check items are strict_v1-CORRECT "
      f"(of {len(correct_idx)} such items)")

run       300 rows, model=['Qwen/Qwen2.5-VL-3B-Instruct']
spotcheck 40 items, 4 known false passes


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

...

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]


sample order matches the CSV on all 300 rows -- images are index-aligned


strict_v1:   0%|          | 0/300 [00:00<?, ?it/s]

all 40 spot-check items are strict_v1-CORRECT (of 141 such items)


In [3]:
# Contact sheets, same format as notebook 17 section 6.
import matplotlib.pyplot as plt

SHEET_DIR = f"{PROJECT_DIR}/figures/spotcheck_strict_v1_correct"
os.makedirs(SHEET_DIR, exist_ok=True)


def caption_for(row):
    """Short label under each page.

    Carries what a coder needs to spot a FALSE PASS without leaving the
    sheet: the two labels that were compared, the entropy, and whether an
    automated rule already flags the item.
    """
    i = int(row["item"])
    s = scored.loc[i]
    flag = "   *** KNOWN FALSE PASS ***" if row["known_false_pass"] else ""
    return (f"item {i}   H={s['perception_entropy']:.2f}   "
            f"strict_v1=CORRECT{flag}\n"
            f"model: {str(s['majority_label'])[:38]}\n"
            f"truth: {str(s['gt_label'])[:38]}")


items = spot["item"].astype(int).tolist()
figs = pilot.plotting.contact_sheet(
    [sample[i]["image"] for i in items],
    [caption_for(r) for _, r in spot.iterrows()],
    ncols=3, per_page=12,
    title="strict_v1 scored these CORRECT - read for FALSE PASSES")

written = []
for page, fig in enumerate(figs, 1):
    path = f"{SHEET_DIR}/spotcheck_correct_p{page}.png"
    fig.savefig(path, dpi=150, facecolor=fig.get_facecolor())
    plt.close(fig)
    written.append(path)
    print(f"  wrote {path}")

# Ship the coding sheet next to the images so the pass is confirm-or-correct.
sheet_path = f"{SHEET_DIR}/coding_sheet.csv"
spot.to_csv(sheet_path, index=False)
print(f"\n{len(items)} items -> {len(figs)} page(s)")
print(f"coding sheet -> {sheet_path}")
print("open: My Drive > uncertainty-math-vlm > figures > spotcheck_strict_v1_correct")
print("\nFill in final_label per item:")
print("  true_pass       - the model really did get it right")
print("  false_pass      - scored correct but the model is wrong (the thing we are hunting)")
print("  needs_visual    - cannot decide from the page")

  wrote /content/drive/MyDrive/uncertainty-math-vlm/figures/spotcheck_strict_v1_correct/spotcheck_correct_p1.png
  wrote /content/drive/MyDrive/uncertainty-math-vlm/figures/spotcheck_strict_v1_correct/spotcheck_correct_p2.png
  wrote /content/drive/MyDrive/uncertainty-math-vlm/figures/spotcheck_strict_v1_correct/spotcheck_correct_p3.png
  wrote /content/drive/MyDrive/uncertainty-math-vlm/figures/spotcheck_strict_v1_correct/spotcheck_correct_p4.png

40 items -> 4 page(s)
coding sheet -> /content/drive/MyDrive/uncertainty-math-vlm/figures/spotcheck_strict_v1_correct/coding_sheet.csv
open: My Drive > uncertainty-math-vlm > figures > spotcheck_strict_v1_correct

Fill in final_label per item:
  true_pass       - the model really did get it right
  false_pass      - scored correct but the model is wrong (the thing we are hunting)
  needs_visual    - cannot decide from the page
